In [0]:
%python
#Getting state code from widget
state = dbutils.widgets.get("state")
print(f"Running for state: {state}")


In [0]:
%python
# Define the catalog and schema
spark.sql("USE CATALOG new_sample")
spark.sql("USE SCHEMA retail_data")

# Reading and transforming data from master_customers_orders_silver table
## Transforming order data from unix timestamp to date and adding is_large_order column

df = spark.sql(f"""
    SELECT *,
           TO_DATE(FROM_UNIXTIME(order_datetime)) AS order_date,
           CASE WHEN number_of_line_items > 2 THEN true ELSE false END AS is_large_order
    FROM new_sample.retail_data.customer_orders_silver
    WHERE state = '{state}'
""")

# Save to UC table, one per state specified in the loop:
table_name = f"customers_orders_{state.replace(' ', '_').lower()}_silver"

display(df)
print(f"Saving to table {table_name}")

In [0]:
%python
df.write.mode("overwrite").saveAsTable(table_name)